# Electrification Scenario Comparison Analysis

This notebook compares electricity consumption patterns across different electrification scenarios to show how appliance electrification changes energy usage patterns.

## Scenarios Analyzed:
- **Baseline**: Gas heating, cooking, water heating
- **Heat Pump**: Electric heating, gas cooking/water heating  
- **Induction Stove**: Electric cooking, gas heating/water heating
- **Heat Pump + Induction**: Electric heating/cooking, gas water heating
- **Full Electric**: All appliances electrified

## Data Integration:
Based on step7_combine_real_and_simulated_electricity_loads.py, which combines:
- Real electricity loads from ResStock data
- Simulated electric appliance loads (heat pump, induction stove, electric water heater)
- Gas loads converted to kWh equivalent for comparison

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set up plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 10

print("Libraries imported successfully!")

In [ ]:
# Configuration
BASE_DIR = Path('data/loadprofiles')
HOUSING_TYPE = 'single-family-detached'
COUNTY = 'alameda'  # County slug (lowercase, dashes for spaces)

# Define scenarios to compare
SCENARIOS = {
    'baseline': 'Baseline (All Gas)',
    'heat_pump': 'Heat Pump Only', 
    'induction_stove': 'Induction Stove Only',
    'heat_pump_and_induction_stove': 'Heat Pump + Induction',
    'heat_pump_and_induction_stove_and_water_heating': 'Full Electric'
}

# Check which scenarios have data
available_scenarios = {}
for scenario_key, scenario_name in SCENARIOS.items():
    data_dir = BASE_DIR / scenario_key / HOUSING_TYPE / COUNTY
    if data_dir.exists():
        available_scenarios[scenario_key] = scenario_name
        
print(f"Analysis Configuration:")
print(f"  County: {COUNTY.title()}")
print(f"  Available scenarios: {len(available_scenarios)}")
for key, name in available_scenarios.items():
    print(f"    - {key}: {name}")

In [ ]:
# Color scheme for different energy sources
ENERGY_SOURCE_COLORS = {
    'Electric - Existing': '#4ECDC4',
    'Electric - Heat Pump': '#FF8E53',
    'Electric - Induction': '#DDA0DD', 
    'Electric - Water Heater': '#96CEB4',
    'Gas - Heating': '#FF6B6B',
    'Gas - Hot Water': '#45B7D1',
    'Gas - Cooking': '#FFEAA7'
}

def load_scenario_data(scenario):
    """
    Load and process all energy data for a given scenario.
    Returns a dictionary with categorized consumption data.
    """
    data_dir = BASE_DIR / scenario / HOUSING_TYPE / COUNTY
    
    electricity_file = data_dir / f'electricity_loads_{COUNTY}.csv'
    gas_file = data_dir / f'gas_loads_{COUNTY}.csv' 
    simulated_file = data_dir / f'electricity_loads_simulated_{COUNTY}.csv'
    
    scenario_data = {}
    
    # Load existing electricity (always present)
    if electricity_file.exists():
        df = pd.read_csv(electricity_file, parse_dates=['timestamp'])
        df.set_index('timestamp', inplace=True)
        
        # Aggregate existing electric appliances (excluding heating/cooking/water heating)
        existing_electric = df[[
            'out.electricity.ceiling_fan.energy_consumption',
            'out.electricity.clothes_dryer.energy_consumption', 
            'out.electricity.dishwasher.energy_consumption',
            'out.electricity.freezer.energy_consumption',
            'out.electricity.lighting_garage.energy_consumption',
            'out.electricity.lighting_interior.energy_consumption',
            'out.electricity.mech_vent.energy_consumption',
            'out.electricity.permanent_spa_heat.energy_consumption',
            'out.electricity.permanent_spa_pump.energy_consumption',
            'out.electricity.plug_loads.energy_consumption',
            'out.electricity.pool_heater.energy_consumption',
            'out.electricity.pool_pump.energy_consumption',
            'out.electricity.refrigerator.energy_consumption'
        ]].sum(axis=1)
        
        scenario_data['Electric - Existing'] = existing_electric
    
    # Load gas data
    if gas_file.exists():
        df = pd.read_csv(gas_file, parse_dates=['timestamp'])
        df.set_index('timestamp', inplace=True)
        df = df.resample('H').sum()  # Resample to hourly
        
        # Gas appliances (check which are still gas in this scenario)
        gas_cols = {
            'Gas - Heating': 'out.natural_gas.heating.energy_consumption.gas.building_avg.kwh',
            'Gas - Hot Water': 'out.natural_gas.hot_water.energy_consumption.gas.building_avg.kwh', 
            'Gas - Cooking': 'out.natural_gas.range_oven.energy_consumption.gas.building_avg.kwh'
        }
        
        for name, col in gas_cols.items():
            if col in df.columns:
                scenario_data[name] = df[col]
    
    # Load simulated electric appliances
    if simulated_file.exists():
        df = pd.read_csv(simulated_file, parse_dates=['timestamp'])
        df.set_index('timestamp', inplace=True)
        df = df.resample('H').sum()  # Resample to hourly
        
        # Simulated electric appliances
        sim_cols = {
            'Electric - Heat Pump': 'simulated.electricity.heat_pump.energy_consumption.electricity.kwh',
            'Electric - Induction': 'simulated.electricity.induction_stove.energy_consumption.electricity.kwh',
            'Electric - Water Heater': 'simulated.electricity.hot_water.energy_consumption.electricity.kwh'
        }
        
        for name, col in sim_cols.items():
            if col in df.columns and df[col].sum() > 0:
                scenario_data[name] = df[col]
    
    return pd.DataFrame(scenario_data)

print("Data loading function defined.")

In [ ]:
# Load data for all available scenarios
scenario_datasets = {}
scenario_totals = {}

for scenario_key, scenario_name in available_scenarios.items():
    print(f"Loading data for {scenario_name}...")
    data = load_scenario_data(scenario_key)
    scenario_datasets[scenario_key] = data
    
    # Calculate annual totals
    annual_totals = data.sum().sort_values(ascending=False)
    scenario_totals[scenario_key] = annual_totals
    
    print(f"  Total annual consumption: {annual_totals.sum():,.0f} kWh")
    print(f"  Categories: {list(annual_totals[annual_totals > 0].index)}")
    
print(f"\nLoaded data for {len(scenario_datasets)} scenarios.")

In [ ]:
# 1. Scenario comparison - Total consumption by energy source
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))

# Left plot: Stacked bar chart by scenario
scenario_matrix = []
scenario_labels = []
all_categories = set()

for scenario_key, scenario_name in available_scenarios.items():
    scenario_labels.append(scenario_name)
    totals = scenario_totals[scenario_key]
    all_categories.update(totals.index)
    scenario_matrix.append(totals)

# Create DataFrame for plotting
all_categories = sorted(list(all_categories))
comparison_df = pd.DataFrame(scenario_matrix, index=scenario_labels, columns=all_categories).fillna(0)

# Plot stacked bars
bottom = np.zeros(len(comparison_df))
for category in all_categories:
    if category in comparison_df.columns:
        color = ENERGY_SOURCE_COLORS.get(category, '#BDC3C7')
        ax1.bar(range(len(comparison_df)), comparison_df[category], 
                bottom=bottom, label=category, color=color)
        bottom += comparison_df[category]

ax1.set_xticks(range(len(comparison_df)))
ax1.set_xticklabels(comparison_df.index, rotation=45, ha='right')
ax1.set_ylabel('Annual Consumption (kWh)')
ax1.set_title('Total Energy Consumption by Scenario')
ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left')

# Right plot: Electric vs Gas split
electric_totals = []
gas_totals = []

for scenario_key in available_scenarios.keys():
    totals = scenario_totals[scenario_key]
    electric = totals[totals.index.str.contains('Electric')].sum()
    gas = totals[totals.index.str.contains('Gas')].sum()
    electric_totals.append(electric)
    gas_totals.append(gas)

x = np.arange(len(scenario_labels))
width = 0.35

ax2.bar(x - width/2, electric_totals, width, label='Electric', color='#4ECDC4')
ax2.bar(x + width/2, gas_totals, width, label='Gas', color='#FF6B6B')

ax2.set_xticks(x)
ax2.set_xticklabels(scenario_labels, rotation=45, ha='right')
ax2.set_ylabel('Annual Consumption (kWh)')
ax2.set_title('Electric vs Gas Consumption')
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
# 2. Monthly consumption patterns across scenarios
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for i, (scenario_key, scenario_name) in enumerate(available_scenarios.items()):
    if i >= len(axes):
        break
        
    data = scenario_datasets[scenario_key]
    monthly_data = data.resample('M').sum()
    monthly_data.index = monthly_data.index.strftime('%b')
    
    # Stacked bar chart for this scenario
    bottom = np.zeros(len(monthly_data))
    for category in data.columns:
        color = ENERGY_SOURCE_COLORS.get(category, '#BDC3C7')
        axes[i].bar(monthly_data.index, monthly_data[category], 
                   bottom=bottom, color=color, label=category)
        bottom += monthly_data[category]
    
    axes[i].set_title(f'{scenario_name}\n({data.sum().sum():,.0f} kWh/year)')
    axes[i].set_ylabel('Monthly Consumption (kWh)')
    
    if i == 0:  # Only show legend on first plot
        axes[i].legend(bbox_to_anchor=(1.05, 1), loc='upper left')

# Hide unused subplots
for i in range(len(available_scenarios), len(axes)):
    axes[i].set_visible(False)

plt.suptitle(f'Monthly Consumption Patterns by Scenario\n{COUNTY.replace("-", " ").title()} County', fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# 3. Hourly load profiles comparison
fig, ax = plt.subplots(figsize=(14, 8))

colors = plt.cm.Set1(np.linspace(0, 1, len(available_scenarios)))

for i, (scenario_key, scenario_name) in enumerate(available_scenarios.items()):
    data = scenario_datasets[scenario_key]
    hourly_profile = data.groupby(data.index.hour).mean().sum(axis=1)
    
    ax.plot(hourly_profile.index, hourly_profile.values, 
            marker='o', linewidth=2.5, label=scenario_name, color=colors[i])

ax.set_xlabel('Hour of Day')
ax.set_ylabel('Average Hourly Consumption (kWh)')
ax.set_title(f'Daily Load Profiles by Scenario\n{COUNTY.replace("-", " ").title()} County')
ax.set_xticks(range(0, 24, 2))
ax.grid(True, alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# 4. Peak demand analysis
peak_demands = {}
total_consumption = {}

for scenario_key, scenario_name in available_scenarios.items():
    data = scenario_datasets[scenario_key]
    hourly_total = data.sum(axis=1)
    
    peak_demands[scenario_name] = hourly_total.max()
    total_consumption[scenario_name] = hourly_total.sum()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Peak demand comparison
scenarios = list(peak_demands.keys())
peaks = list(peak_demands.values())

bars1 = ax1.bar(scenarios, peaks, color=colors[:len(scenarios)])
ax1.set_ylabel('Peak Demand (kW)')
ax1.set_title('Peak Hourly Demand by Scenario')
ax1.tick_params(axis='x', rotation=45)

# Add value labels
for bar, value in zip(bars1, peaks):
    ax1.text(bar.get_x() + bar.get_width()/2., bar.get_height() + bar.get_height()*0.01,
             f'{value:.2f}', ha='center', va='bottom')

# Total consumption comparison  
totals = list(total_consumption.values())
bars2 = ax2.bar(scenarios, totals, color=colors[:len(scenarios)])
ax2.set_ylabel('Total Annual Consumption (kWh)')
ax2.set_title('Total Annual Consumption by Scenario')
ax2.tick_params(axis='x', rotation=45)

# Add value labels
for bar, value in zip(bars2, totals):
    ax2.text(bar.get_x() + bar.get_width()/2., bar.get_height() + bar.get_height()*0.01,
             f'{value:,.0f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

# Print comparison stats
baseline_peak = peak_demands.get('Baseline (All Gas)', 0)
baseline_total = total_consumption.get('Baseline (All Gas)', 0)

print("Impact of Electrification:")
for scenario, peak in peak_demands.items():
    if scenario != 'Baseline (All Gas)' and baseline_peak > 0:
        peak_change = ((peak - baseline_peak) / baseline_peak) * 100
        total_change = ((total_consumption[scenario] - baseline_total) / baseline_total) * 100
        print(f"  {scenario}:")
        print(f"    Peak demand change: {peak_change:+.1f}%")
        print(f"    Total consumption change: {total_change:+.1f}%")

In [ ]:
# 5. Seasonal heating analysis (winter vs summer patterns)
winter_months = [12, 1, 2]
summer_months = [6, 7, 8]

seasonal_analysis = {}

for scenario_key, scenario_name in available_scenarios.items():
    data = scenario_datasets[scenario_key]
    data_with_month = data.copy()
    data_with_month['month'] = data_with_month.index.month
    
    winter_avg = data_with_month[data_with_month['month'].isin(winter_months)].drop('month', axis=1).mean().sum()
    summer_avg = data_with_month[data_with_month['month'].isin(summer_months)].drop('month', axis=1).mean().sum()
    
    seasonal_analysis[scenario_name] = {
        'Winter': winter_avg,
        'Summer': summer_avg,
        'Ratio': winter_avg / summer_avg if summer_avg > 0 else 0
    }

# Plot seasonal comparison
scenarios = list(seasonal_analysis.keys())
winter_values = [seasonal_analysis[s]['Winter'] for s in scenarios]
summer_values = [seasonal_analysis[s]['Summer'] for s in scenarios]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

x = np.arange(len(scenarios))
width = 0.35

ax1.bar(x - width/2, winter_values, width, label='Winter (Dec-Feb)', color='#87CEEB')
ax1.bar(x + width/2, summer_values, width, label='Summer (Jun-Aug)', color='#FFA07A')

ax1.set_xticks(x)
ax1.set_xticklabels(scenarios, rotation=45, ha='right')
ax1.set_ylabel('Average Hourly Consumption (kWh)')
ax1.set_title('Seasonal Consumption Patterns')
ax1.legend()

# Winter/summer ratios
ratios = [seasonal_analysis[s]['Ratio'] for s in scenarios]
bars = ax2.bar(scenarios, ratios, color=colors[:len(scenarios)])
ax2.set_ylabel('Winter/Summer Ratio')
ax2.set_title('Seasonal Variation by Scenario')
ax2.tick_params(axis='x', rotation=45)
ax2.axhline(y=1.0, color='red', linestyle='--', alpha=0.7, label='Equal consumption')
ax2.legend()

# Add value labels
for bar, value in zip(bars, ratios):
    ax2.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
             f'{value:.2f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

print("\nSeasonal Analysis:")
for scenario, data in seasonal_analysis.items():
    print(f"  {scenario}: Winter/Summer ratio = {data['Ratio']:.2f}")

In [ ]:
# 6. Electrification impact summary
print("=" * 80)
print("ELECTRIFICATION SCENARIO ANALYSIS SUMMARY")
print("=" * 80)
print(f"\nCounty: {COUNTY.replace('-', ' ').title()}")
print(f"Housing Type: {HOUSING_TYPE.replace('-', ' ').title()}")

# Get baseline for comparison
baseline_scenario = None
for key, name in available_scenarios.items():
    if 'baseline' in key.lower():
        baseline_scenario = key
        break

if baseline_scenario:
    baseline_total = scenario_totals[baseline_scenario].sum()
    baseline_electric = scenario_totals[baseline_scenario][scenario_totals[baseline_scenario].index.str.contains('Electric')].sum()
    baseline_gas = scenario_totals[baseline_scenario][scenario_totals[baseline_scenario].index.str.contains('Gas')].sum()
    
    print(f"\nBaseline Scenario ({available_scenarios[baseline_scenario]}):")
    print(f"  Total consumption: {baseline_total:,.0f} kWh/year")
    print(f"  Electric: {baseline_electric:,.0f} kWh ({baseline_electric/baseline_total*100:.1f}%)")
    print(f"  Gas: {baseline_gas:,.0f} kWh ({baseline_gas/baseline_total*100:.1f}%)")
    
    print(f"\nElectrification Impact:")
    for scenario_key, scenario_name in available_scenarios.items():
        if scenario_key != baseline_scenario:
            total = scenario_totals[scenario_key].sum()
            electric = scenario_totals[scenario_key][scenario_totals[scenario_key].index.str.contains('Electric')].sum()
            gas = scenario_totals[scenario_key][scenario_totals[scenario_key].index.str.contains('Gas')].sum()
            
            total_change = ((total - baseline_total) / baseline_total) * 100
            electric_change = ((electric - baseline_electric) / baseline_electric) * 100 if baseline_electric > 0 else 0
            gas_change = ((gas - baseline_gas) / baseline_gas) * 100 if baseline_gas > 0 else -100
            
            print(f"\n  {scenario_name}:")
            print(f"    Total change: {total_change:+.1f}% ({total:,.0f} kWh/year)")
            print(f"    Electric change: {electric_change:+.1f}% ({electric:,.0f} kWh/year)")
            print(f"    Gas change: {gas_change:+.1f}% ({gas:,.0f} kWh/year)")
            print(f"    Electrification ratio: {electric/total*100:.1f}% electric")

# Peak demand analysis
print(f"\n\nPeak Demand Analysis:")
if baseline_scenario and baseline_scenario in [k for k in available_scenarios.keys()]:
    baseline_peak = peak_demands[available_scenarios[baseline_scenario]]
    print(f"  Baseline peak: {baseline_peak:.2f} kW")
    
    for scenario_name, peak in peak_demands.items():
        if scenario_name != available_scenarios[baseline_scenario]:
            change = ((peak - baseline_peak) / baseline_peak) * 100
            print(f"  {scenario_name}: {peak:.2f} kW ({change:+.1f}% vs baseline)")

print("\n" + "=" * 80)
print(f"Analysis complete! Compared {len(available_scenarios)} scenarios.")
print("=" * 80)

In [ ]:
# 7. Export comparison data
output_dir = Path('scenario_comparison_results')
output_dir.mkdir(exist_ok=True)

# Export scenario totals comparison
comparison_summary = comparison_df.T  # Transpose so categories are rows
comparison_file = output_dir / f'scenario_comparison_{COUNTY}.csv'
comparison_summary.to_csv(comparison_file)

# Export peak demand analysis
peak_analysis = pd.DataFrame({
    'Scenario': list(peak_demands.keys()),
    'Peak_Demand_kW': list(peak_demands.values()),
    'Total_Annual_kWh': list(total_consumption.values())
})
peak_file = output_dir / f'peak_analysis_{COUNTY}.csv'
peak_analysis.to_csv(peak_file, index=False)

# Export seasonal analysis
seasonal_df = pd.DataFrame(seasonal_analysis).T
seasonal_file = output_dir / f'seasonal_analysis_{COUNTY}.csv'
seasonal_df.to_csv(seasonal_file)

print(f"Scenario comparison results exported to:")
print(f"  Consumption comparison: {comparison_file}")
print(f"  Peak demand analysis: {peak_file}")
print(f"  Seasonal analysis: {seasonal_file}")